In [2]:
import numpy as np
import pandas as pd
from shapely.geometry import Polygon

# ── Settings ─────────────────────────────────────────────────────────────────

INPUT_PATH  = "./top_buyers_us_oil_2025.csv"
OUTPUT_PATH = "./top_buyers_spatial.csv"

HALF      = 10.0  # Circle radius in degrees
N_PTS     = 24    # Points per side (more points → smoother curves)
COUNTRY_GAP = 0.005  # The gap between the polygons of COUNTRY (in units of a squircle)
REGION_GAP  = 0.1   # Spacing for REGION polygons — 0=no spacing

# ── Squarified Treemap ────────────────────────────────────────────────────────

def _worst(row, w):
    s = sum(row)
    if s == 0 or min(row) == 0 or w == 0:
        return float("inf")
    return max(w * w * max(row) / (s * s), s * s / (w * w * min(row)))


def _place_row(row, x, y, dx, dy, landscape, gap):
    s = sum(row)
    rects = []
    if landscape:
        h = s / dx if dx > 0 else dy
        cx = x
        for sz in row:
            w = sz / h if h > 0 else 0
            rects.append((cx + gap, y + gap, cx + w - gap, y + h - gap))
            cx += w
    else:
        w = s / dy if dy > 0 else dx
        cy = y
        for sz in row:
            h = sz / w if w > 0 else 0
            rects.append((x + gap, cy + gap, x + w - gap, cy + h - gap))
            cy += h
    return rects


def _squarify(sizes, x, y, dx, dy, gap):
    if not sizes or dx < 1e-9 or dy < 1e-9:
        return [(x, y, x, y)] * len(sizes)
    if len(sizes) == 1:
        return [(x + gap, y + gap, x + dx - gap, y + dy - gap)]

    landscape = dx >= dy
    w = dx if landscape else dy

    n = 1
    for i in range(1, len(sizes)):
        if _worst(sizes[:i + 1], w) <= _worst(sizes[:i], w):
            n = i + 1
        else:
            break

    row  = sizes[:n]
    rest = sizes[n:]
    s    = sum(row)

    rects = _place_row(row, x, y, dx, dy, landscape, gap)

    if landscape:
        h = s / dx if dx > 0 else 0
        rects += _squarify(rest, x, y + h, dx, max(0.0, dy - h), gap)
    else:
        w2 = s / dy if dy > 0 else 0
        rects += _squarify(rest, x + w2, y, max(0.0, dx - w2), dy, gap)

    return rects


def squarified_layout(values, x0, y0, x1, y1, gap=0.0):
    values = [float(v) for v in values]
    n      = len(values)
    total  = sum(values)
    area   = (x1 - x0) * (y1 - y0)

    if total == 0 or n == 0:
        return [(x0, y0, x1, y1)] * n

    order       = sorted(range(n), key=lambda i: -values[i])
    norm_sorted = [values[i] / total * area for i in order]

    rects_sorted = _squarify(norm_sorted, x0, y0, x1 - x0, y1 - y0, gap)

    result = [None] * n
    for si, oi in enumerate(order):
        result[oi] = rects_sorted[si]
    return result


# ── Squircle mapping ──────────────────────────────────────────────────────────

def squircle(x, y, half=HALF):
    xn = float(np.clip(x / half, -1.0, 1.0))
    yn = float(np.clip(y / half, -1.0, 1.0))
    u  = xn * float(np.sqrt(max(0.0, 1.0 - yn ** 2 / 2.0)))
    v  = yn * float(np.sqrt(max(0.0, 1.0 - xn ** 2 / 2.0)))
    return u * half, -v * half


def rect_to_polygon(x0, y0, x1, y1, n_pts=N_PTS, half=HALF):
    pts = []
    for t in np.linspace(0, 1, n_pts, endpoint=False):
        pts.append((x0 + t * (x1 - x0), y0))
    for t in np.linspace(0, 1, n_pts, endpoint=False):
        pts.append((x1, y0 + t * (y1 - y0)))
    for t in np.linspace(0, 1, n_pts, endpoint=False):
        pts.append((x1 - t * (x1 - x0), y1))
    for t in np.linspace(0, 1, n_pts, endpoint=False):
        pts.append((x0, y1 - t * (y1 - y0)))

    mapped = [squircle(px, py, half) for px, py in pts]
    poly   = Polygon(mapped)
    if not poly.is_valid:
        poly = poly.buffer(0)
    return poly


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    df = pd.read_csv(INPUT_PATH)
    df["Share_Total"] = df["Share_Total"].round(6)

    # ── Regions ────────────────────────────────────────────────────────
    region_order  = list(df["Region"].unique())
    region_shares = (
        df.groupby("Region", sort=False)["Share_Total"]
        .sum()
        .reindex(region_order)
    )

    reg_rects_raw = squarified_layout(
    region_shares.values, -HALF, -HALF, HALF, HALF, gap=REGION_GAP
    )
    region_rect_map = {r: reg_rects_raw[i] for i, r in enumerate(region_order)}
    
    g = REGION_GAP

    # The boundary X between the left and right halves
    left_regions_all = ["North America", "Central & South America", "World", "Africa"]
    right_total  = region_shares["Europe"] + region_shares["Asia-Pacific"]
    left_total   = sum(region_shares[r] for r in left_regions_all)
    total        = left_total + right_total
    x_split      = -HALF + (left_total / total) * 2 * HALF
    
    # Left side: top and bottom are separate
    top_left_regions = ["North America", "Central & South America"]
    bot_left_regions = ["World", "Africa"]
    
    top_left_shares = [region_shares[r] for r in top_left_regions]
    bot_left_shares = [region_shares[r] for r in bot_left_regions]
    
    top_left_total = sum(top_left_shares)
    bot_left_total = sum(bot_left_shares)
    
    y_split_left = -HALF + (top_left_total / (top_left_total + bot_left_total)) * 2 * HALF

    
    # World + Africa
    bot_left_rects = squarified_layout(
    bot_left_shares,
    -HALF, y_split_left, x_split, HALF,
    gap=REGION_GAP
    )
    top_left_rects = squarified_layout(
        top_left_shares,
        -HALF, -HALF, x_split, y_split_left,
        gap=REGION_GAP
    )
    
    region_rect_map = {}
    for i, r in enumerate(top_left_regions):
        region_rect_map[r] = top_left_rects[i]
    for i, r in enumerate(bot_left_regions):
        region_rect_map[r] = bot_left_rects[i]

    europe_share = region_shares["Europe"]
    y_split      = -HALF + (europe_share / right_total) * 2 * HALF
    region_rect_map["Europe"]       = (x_split + g, -HALF + g,   HALF - g, y_split - g)
    region_rect_map["Asia-Pacific"] = (x_split + g,  y_split + g, HALF - g, HALF - g)

    region_poly = {}
    for region in region_order:
        rr = region_rect_map[region]
        poly = rect_to_polygon(*rr) 
        if not poly.is_valid:
            poly = poly.buffer(0)
        region_poly[region] = poly

    for r, p in region_poly.items():
        geom_type = p.geom_type
        area      = p.area

    # ── Countries—inside the region's rectangle, with a gap ──────────────
    country_poly = {}

    for region in region_order:
        rr   = region_rect_map[region]
        mask = df["Region"] == region
        sub  = df[mask]
        if sub.empty:
            continue

        c_rects = squarified_layout(
            sub["Share_Total"].values,
            rr[0], rr[1], rr[2], rr[3],
            gap=COUNTRY_GAP,
        )

        for i, (idx, _) in enumerate(sub.iterrows()):
            poly = rect_to_polygon(*c_rects[i])
            country_poly[idx] = poly

    # ── Column entry ─────────────────────────────────────────────────
    df["region_polygon"]  = ""
    df["country_polygon"] = ""
    df["country_center"]  = ""

    for idx, row in df.iterrows():
        region = row["Region"]

        if idx in country_poly:
            poly = country_poly[idx]
            df.at[idx, "country_polygon"] = poly.wkt
            c = poly.centroid
            df.at[idx, "country_center"] = f"POINT ({c.x:.6f} {c.y:.6f})"

        if region in region_poly:
            df.at[idx, "region_polygon"] = region_poly[region].wkt
    # ── Regions Centers ────────────────────────────────────────────────
    df["region_center"] = df["Region"].map(
        {r: f"POINT ({region_poly[r].centroid.x:.6f} {region_poly[r].centroid.y:.6f})"
         for r in region_poly}
    )
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"\n✓ Saved: {OUTPUT_PATH}")
    print(f"  Columns: {list(df.columns)}")


if __name__ == "__main__":
    main()


✓ Saved: ./top_buyers_spatial.csv
  Columns: ['Rank', 'Country', 'Region', 'Value', 'Share_Total', 'region_polygon', 'country_polygon', 'country_center', 'region_center']
